In [4]:
from slither.slither import Slither
import os
from dotenv import load_dotenv
import sys
sys.path.append("../")

load_dotenv()
from src.detectors import IVC
from src.response.IVC_response import build_detector_response

In [5]:
def run_IVC_module(chain: str, address: str) -> dict:

    prefix=''
    if chain=="arbitrum": prefix = 'arbi:'
    if chain=="base": prefix = 'base:'
    if chain=='optimism': prefix = 'optim:'

    try:
        sl = Slither(prefix+address, etherscan_api_key=os.getenv("ETHERSCAN_API_KEY"))
    except Exception as e :
        print(e)
        raise(e)

    ACM_results = IVC.run(sl)

    response_AC = build_detector_response(chain, address, ACM_results )
  
    return ACM_results, response_AC

In [6]:
# run on local file : 
! solc-select use 0.8.20
result, response = run_IVC_module(chain='local', address='../test-contracts/IVC_vulnerable.sol')

print(result)

response

Switched global version to 0.8.20
IVCVulnerable.delegateCallTo(address,bytes)
❌ unvalidated tainted/inp var in calls : target
✨RESULT:  {'IVCVulnerable.delegateCallTo(address,bytes)': [('IVCVulnerable.delegateCallTo(address,bytes).target', ['target.delegatecall(data)'])]}
vars_to_check(IVCVulnerable.delegateCallTo(address,bytes)) : ['ret', 'ok', 'target', 'data']

IVCVulnerable.lowLevelCallTo(address,uint256,bytes)
❌ unvalidated tainted/inp var in calls : target
✨RESULT:  {'IVCVulnerable.delegateCallTo(address,bytes)': [('IVCVulnerable.delegateCallTo(address,bytes).target', ['target.delegatecall(data)'])], 'IVCVulnerable.lowLevelCallTo(address,uint256,bytes)': [('IVCVulnerable.lowLevelCallTo(address,uint256,bytes).target', ['target.call{value:value}(data)'])]}
vars_to_check(IVCVulnerable.lowLevelCallTo(address,uint256,bytes)) : ['ret', 'ok', 'target', 'value', 'data']

IVCVulnerable.staticCallTo(address,bytes)
❌ unvalidated tainted/inp var in calls : target
✨RESULT:  {'IVCVulnerable.de

{'chain': 'local',
 'address': '../test-contracts/IVC_vulnerable.sol',
 'issues_found': [{'ID': 'IVC-001',
   'Type': 'Input-Validation-Calls',
   'Category': 'Dangerous delegatecall without input validation',
   'Title': 'IVC-001 — Missing Input Validation in `delegateCallTo()` leading to unsafe calls by `target`',
   'Severity': 'HIGH',
   'Description': 'In `IVCVulnerable.delegateCallTo(address,bytes)`, \nvariable `IVCVulnerable.delegateCallTo(address,bytes).target` is an user-input value (or) user-input derived value which is used as a call_target address without proper validation.\nThis might lead to dangerous user-controlled target calls.\n\nObserved call sites:\n    •  target.delegatecall(data)\n\n',
   'Function': 'IVCVulnerable.delegateCallTo(address,bytes)',
   'Variable': 'IVCVulnerable.delegateCallTo().target'},
  {'ID': 'IVC-003',
   'Type': 'Input-Validation-Calls',
   'Category': 'Low-level external call without input validation',
   'Title': 'IVC-003 — Missing Input Val